In [1]:
import pymupdf
from langchain_core.documents import Document
from transformers import CLIPProcessor,CLIPModel
from PIL import Image
import torch
import numpy as np 
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_480/3705220676.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [ ]:
def embed_image(image_data):
    # embed image using CLIP
    if isinstance(image_data, str):          # if path
        image = Image.open(image_data).convert("RGB")
    else:                                     # if PIL image
        image = image_data

    inputs = clip_processor(images=image, return_tensors="pt")

    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)

    #  transformers >= 5.0 returns BaseModelOutputWithPooling
    if hasattr(features, "image_embeds"):
        features = features.image_embeds
    elif hasattr(features, "pooler_output"):
        features = features.pooler_output

    # Normalize to unit vector
    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy().reshape(-1)


# ---------- Embed Text ----------
def embed_text(text):
    # Embed text using CLIP
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    )

    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)

    #  transformers >= 5.0 returns BaseModelOutputWithPooling
    if hasattr(features, "text_embeds"):
        features = features.text_embeds
    elif hasattr(features, "pooler_output"):
        features = features.pooler_output

    # Normalize to unit vector
    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy().reshape(-1)

In [10]:
# process pdf
pdf_path = "multimodal_sample.pdf"
doc = pymupdf.open(pdf_path)

# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {}
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

for i,page in enumerate(doc):
    # process text
    text = page.get_text()
    if text.strip():
        temp_doc = Document(page_content=text, metadata={"page":i, "type":text})
        text_chunks = splitter.split_documents([temp_doc])
        for chunk in text_chunks:
            embedding = embed_text(
                chunk.page_content
            )
            all_embeddings.append(embedding)
            all_docs.append(chunk)

In [13]:
# Process images
# Three important actions
# convert pdf images to PIL
# Store as base64 for GPT - 4v
# create CLIP embedding for retrieval
for img_index, img in enumerate(page.get_images(full=True)):
    try:
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]

        # convert to PIL image
        pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        # create unique identifier
        image_id = f"page_{i}_img_{img_index}"

        # store image as base64 for later use
        buffered = io.BytesIO()
        pil_image.save(buffered, format="PNG")
        img_base64 = base64.b64encode(buffered.getvalue()).decode()
        image_data_store[image_id] = img_base64

        # Embed image using CLIP
        embedding = embed_image(pil_image)
        all_embeddings.append(embedding)

        # create document for image
        image_doc = Document(page_content=f"[Image:{image_id}]",metadata={"page": i, "type":"image","image_id":image_id})
        all_docs.append(image_doc)
    except Exception as e:
        print(f"Error processing image {img_index} on page {i}: {e}")
        continue
doc.close()

In [14]:
all_embeddings

[array([-2.67242664e-03,  1.28299706e-02, -5.18313721e-02,  4.14879657e-02,
        -2.33941898e-02, -7.55865779e-03, -3.67659405e-02,  1.19710669e-01,
         8.52081031e-02,  2.05425336e-03, -1.11534661e-02, -1.29592260e-02,
         5.25014773e-02, -3.65396193e-03,  4.76078391e-02,  1.58372652e-02,
         2.03388110e-02,  4.35362346e-02, -3.29167536e-03,  2.03181617e-02,
         1.88024982e-03, -4.23494056e-02,  5.44098532e-03,  3.70936058e-02,
        -1.65622644e-02,  6.48645032e-03, -4.78011854e-02,  8.67485907e-03,
         5.88859655e-02, -3.21394429e-02,  4.32440080e-02,  9.65300668e-03,
        -4.47922479e-03, -1.94857977e-02, -3.63503210e-02, -1.23471851e-02,
        -2.17929408e-02, -1.99016295e-02,  8.09619874e-02, -3.32986638e-02,
        -2.38901265e-02, -3.96138765e-02, -1.27280103e-02,  3.50380652e-02,
        -2.52217241e-02,  2.00033584e-03,  1.49660408e-02, -2.31976435e-02,
        -6.86791539e-02, -5.25768322e-04, -2.22545974e-02, -1.04103843e-02,
        -1.9

In [15]:
# create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(all_embeddings)

# create custom FAISS index since we have precomputed embeddings
vector_store = FAISS.from_embeddings(
    text_embeddings = [(doc.page_content,emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding = None,
    metadatas = [doc.metadata for doc in all_docs]
)
vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [16]:
from langchain_groq import ChatGroq
llm = ChatGroq(
     model="qwen/qwen3.8-27b", 
     temperature=0.7,
     api_key=os.getenv("GROQ_API_KEY")
 )

In [24]:
def retrieve_multimodal(query, k=5):
    # unified retrieval using CLIP embeddings for both text and images
    # Embed query using CLIP
    query_embedding = embed_text(query)
    # Search in unified vector store
    results = vector_store.similarity_search_by_vector(
        embedding = query_embedding,
        k = k
    )
    return results

In [25]:
def create_multimodal_message(query, retrieved_docs):
    # create a message with both text and images for GPT 4v
    content = []
    # add the query
    content.append({
        "type":"text",
        "text":f"Question:{query}\n\ncontext:\n"
    })
    # separte text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type")=="text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type")=="image"]   

    #add text context
    if text_docs:
        text_context = "\n\n".join([f"page{doc.metadata['page']}: {doc.page_content}"  
        for doc in text_docs])
        content.append({
            "type":"text",
            "text":f"Text excerpts:\n{text_context}\n"
        })
    # add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type":"text",
                "text":f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type":"image_url",
                "image_url":{"url":f"data:image/png;base64,{image_data_store[image_id]}"}
            })
    # add instruction
    content.append({"type":"text",
                   "text":"\n\nPlease answer the question based on the provided text and images"
    })
    return HumanMessage(content=content)   


In [26]:
def mutlimodal_pdf_rag_pipeline(query):
    # Main pipeline for multimodal rag
    # Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)

    # create multimodal message
    message = create_multimodal_message(query,context_docs)

    # Get response
    response = llm.invoke([message])

    # print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")

    for doc in context_docs:
        doc_type = doc.metadata.get("type","unknown")
        page = doc.metadata.get("page","?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"-Text from page {page}:{preview}")
        else:
            print(f"-Image from page {page}")
    
    print("\n")
    return response.content

In [28]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "what visual elements are present in the document?"
        "What are the bar colors?"
    ]
    for query in queries:
        print(f"\nQuery: {query}")
        print("-"*50)
        answer = mutlimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("="*70)


Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------

Retrieved 3 documents:
-Image from page 0
-Image from page 0
-Image from page 0


Answer: Based on the provided image, which appears to be a simple bar chart with three bars of increasing height (blue, green, and red), here is what it shows about revenue trends, assuming the bars represent revenue over time or across categories:

- **Upward Trend:** The most prominent feature is that the bars get progressively taller from left to right. This indicates a consistent increase in the value being measured (assumed to be revenue).
- **Sequential Growth:** If the bars represent consecutive periods (e.g., Year 1, Year 2, Year 3) or sequential categories, the chart shows that revenue grew from the first period/category to the second, and then grew even more significantly from the second to the third.
- **Highest Value on the Right:** The red bar, being the tallest, represents the